In [1]:
import os
import numpy as np
import pandas as pd
import sncosmo
import dynesty
from PIL.ImImagePlugin import DATE
from joblib import Parallel, delayed
from functools import partial

# --- CONFIGURATION ---
SPECTRA_DIR = "data/all_spectra"
PARAM_FILE = os.path.join("data", "cfasnIa_param.dat")
DATA_FILE = os.path.join("data", "cfa_SNID_results.csv")
NLIVE = 100
MODEL_NAME = 'salt3'

In [4]:
df = pd.read_csv(os.path.join('/Users/pxm588@student.bham.ac.uk/PhD/bayesian_framework', DATA_FILE))
print(df.head())

                        Filename     SNR SN_Name  Age_(days)  Age_Unc_(days)  \
0  sn1997cn-19970609.21-fast.flm    8.97  1997cn   21.744689        0.786782   
1  sn1999by-19990513.18-fast.flm  162.82  1999by    1.975654        0.498902   
2   sn2007F-20070126.43-fast.flm   72.99   2007F    2.960141        0.000000   
3  sn2004at-20040323.24-fast.flm   84.06  2004at   -4.656167        0.782549   
4  sn2003hu-20031002.13-fast.flm    9.26  2003hu   12.120930        1.953488   

   redshift  Dm15 Subtype  dm15  bootstrap_age  snid_std_dev  delta_age  \
0    0.0168  1.90    91bg  1.90           25.4        1.4717   3.655311   
1    0.0022  1.97    91bg  1.97            1.0        3.3174  -0.975654   
2    0.0236  0.96       N  0.96            2.0        0.7246  -0.960141   
3    0.0223  1.05       N  1.05           -3.0        0.8591   1.656167   
4    0.0750   NaN     91T   NaN            NaN           NaN        NaN   

   delta_age_unc       MJD  
0       1.668810  50608.21  
1       3.

In [ ]:
def load_flm_spectrum(file_path):
    """Loading an FLM spectrum from datafiles."""
    try:
        data = np.genfromtxt(file_path, invalid_raise=False)
        if len(data.shape) < 2:
            return np.array([]), np.array([]), np.array([])

        wavelength = data[:, 0]
        flux = data[:, 1]

        if data.shape[1] >= 3:
            flux_err = data[:, 2]
            zero_mask = (flux_err <= 0) | (~np.isfinite(flux_err))
            if np.any(zero_mask):
                fallback_err = 0.1 * np.abs(flux)
                min_err = np.percentile(np.abs(flux[flux > 0]), 5) * 0.1 if np.any(flux > 0) else 1e-18
                fallback_err = np.maximum(fallback_err, min_err)
                flux_err[zero_mask] = fallback_err[zero_mask]
        else:
            flux_err = 0.1 * np.abs(flux)

        mask = np.isfinite(flux) & np.isfinite(flux_err) & (flux_err > 0)
        return wavelength[mask], flux[mask], flux_err[mask]
    except Exception:
        return np.array([]), np.array([]), np.array([])

def parse_data_file(file_path):
    params = {}
    pd.read_csv(file_path)
    if not os.path.exists(file_path):
        print(f"Error: Param file {file_path} not found.")
        return params

    return params

In [10]:
def parse_param_csv(file_path):
    """
    Loads the SN metadata CSV and returns a dictionary for lookup.
    """

    params = {}
    if not os.path.exists(file_path):
        print(f"Error: CSV file {file_path} not found.")
        return params

    # Load the CSV
    df = pd.read_csv(file_path)

    for _, row in df.iterrows():
        # Ensure we have a valid SN Name to use as a key
        file_name = str(row['Filename'])

        try:
            # In your old code, mjd_max was the date of peak brightness.
            # In your CSV, we have MJD (obs date) and Age.
            # Peak Date (mjd_max) = MJD_obs - Age
            mjd = row['MJD']

            params[file_name] = {
                'z': float(row['redshift']),
                'mjd': float(mjd),
                'true_age': float(row['Age_(days)']), # Optional: keep for verification
                'redshift': float(row['redshift'])
            }
        except (ValueError, KeyError):
            # Skip rows with missing data (NaN) or formatting errors
            continue

    return params

In [11]:
parse_param_csv(os.path.join('/Users/pxm588@student.bham.ac.uk/PhD/bayesian_framework', DATA_FILE))

{'sn1997cn-19970609.21-fast.flm': {'z': 0.0168,
  'mjd': 50608.21,
  'true_age': 21.744689221086333,
  'redshift': 0.0168},
 'sn1999by-19990513.18-fast.flm': {'z': 0.0022,
  'mjd': 51311.18,
  'true_age': 1.9756535621664353,
  'redshift': 0.0022},
 'sn2007F-20070126.43-fast.flm': {'z': 0.0236,
  'mjd': 54126.43,
  'true_age': 2.9601406799519694,
  'redshift': 0.0236},
 'sn2004at-20040323.24-fast.flm': {'z': 0.0223,
  'mjd': 53087.24,
  'true_age': -4.65616746552092,
  'redshift': 0.0223},
 'sn2003hu-20031002.13-fast.flm': {'z': 0.075,
  'mjd': 52914.13,
  'true_age': 12.120930232557058,
  'redshift': 0.075},
 'sn2001V-20010427.35-fast.flm': {'z': 0.0151,
  'mjd': 52026.35,
  'true_age': 52.75342330804418,
  'redshift': 0.0151},
 'sn1994D-19940503.25-fast.flm': {'z': 0.0029,
  'mjd': 49475.25,
  'true_age': 42.626383487885136,
  'redshift': 0.0029},
 'sn2001N-20010201.37-fast.flm': {'z': 0.021,
  'mjd': 51941.37,
  'true_age': 10.35259549461284,
  'redshift': 0.021},
 'sn2004as-20040422